# Visual Memory Codec on Kaggle

Efficient persistent representations for machine visual memory.

## Fresh session note

- `/kaggle/working` is session-local.
- Manually installed packages are session-local.
- Restarting the Python kernel may temporarily preserve the underlying environment, but a completely new Kaggle session should be assumed clean.
- This notebook is therefore designed to reinstall and reproduce the smoke-test environment from scratch in a fresh Kaggle GPU session.


## 1. Configuration

In [ ]:
import os
from pathlib import Path

REPO_URL = 'https://github.com/PrayYoung/visual-memory-codec.git'
WORK_ROOT = Path('/kaggle/working')
REPO_DIR = WORK_ROOT / 'visual-memory-codec'
CONFIG_PATH = 'configs/natural_coco_smoke.json'
COCO_OUTPUT_DIR = 'data/coco'
SMOKE_SAMPLE_COUNT = 6
EXPORT_ZIP_PATH = WORK_ROOT / 'natural_coco_smoke.zip'
REQUIRE_HF_TOKEN = False
print({'repo_url': REPO_URL, 'repo_dir': str(REPO_DIR), 'config_path': CONFIG_PATH, 'smoke_sample_count': SMOKE_SAMPLE_COUNT})

## 2. Secrets

In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
try:
    GITHUB_TOKEN = secrets.get_secret('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Missing Kaggle secret GITHUB_TOKEN. Add it before running the notebook.') from exc

HF_TOKEN = None
if REQUIRE_HF_TOKEN:
    try:
        HF_TOKEN = secrets.get_secret('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('REQUIRE_HF_TOKEN=True but Kaggle secret HF_TOKEN is missing.') from exc

print('Secrets loaded: GITHUB_TOKEN=yes, HF_TOKEN=' + ('yes' if HF_TOKEN else 'no'))

## 3. Clone or update repository

In [ ]:
import os
import shutil
import subprocess
import textwrap

os.chdir('/kaggle/working')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

askpass_path = WORK_ROOT / 'git_askpass.sh'
askpass_path.write_text(textwrap.dedent('''\
    #!/usr/bin/env bash
    case "$1" in
      *Username*) echo "x-access-token" ;;
      *Password*) echo "$GITHUB_TOKEN" ;;
      *) echo "" ;;
    esac
'''))
askpass_path.chmod(0o700)

env = os.environ.copy()
env['GITHUB_TOKEN'] = GITHUB_TOKEN
env['GIT_ASKPASS'] = str(askpass_path)
env['GIT_TERMINAL_PROMPT'] = '0'

clone = subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], env=env, capture_output=True, text=True)
if clone.returncode != 0:
    raise RuntimeError('GitHub authentication or clone failed:\n' + clone.stderr)

os.chdir(REPO_DIR)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Cloned commit:', commit_hash)

## 4. Install environment

In [ ]:
import os
import subprocess

env = os.environ.copy()
if HF_TOKEN:
    env['HF_TOKEN'] = HF_TOKEN

install = subprocess.run(['bash', 'scripts/install_kaggle_env.sh'], env=env, text=True)
if install.returncode != 0:
    raise RuntimeError('Kaggle environment install failed before benchmark execution.')

In [ ]:
import diffusers
import platform
import safetensors
import timm
import torch
import torchvision
import transformers

print('python', platform.python_version())
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)
print('transformers', transformers.__version__)
print('diffusers', diffusers.__version__)
print('accelerate import ok')
print('safetensors', safetensors.__version__)
print('timm', timm.__version__)

## 5. GPU validation

In [ ]:
import math
import torch

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not available. Stop here and switch the Kaggle notebook runtime to GPU before downloading models.')

device_props = torch.cuda.get_device_properties(0)
gpu_name = torch.cuda.get_device_name(0)
arch_list = torch.cuda.get_arch_list()
cuda_version = torch.version.cuda
total_gb = device_props.total_memory / (1024 ** 3)
capability = f'sm_{device_props.major}{device_props.minor}'

print('torch version:', torch.__version__)
print('torch CUDA version:', cuda_version)
print('GPU name:', gpu_name)
print('GPU memory GB:', round(total_gb, 2))
print('GPU capability:', capability)
print('torch arch list:', arch_list)

if 'Tesla P100' in gpu_name and capability != 'sm_60':
    raise RuntimeError(f'Expected a Tesla P100 with sm_60, but found {gpu_name} / {capability}.')

if capability not in arch_list:
    raise RuntimeError(f'This torch build does not advertise support for {capability}. Installed arch list: {arch_list}')

a = torch.randn((1024, 1024), device='cuda')
b = torch.randn((1024, 1024), device='cuda')
c = a @ b
torch.cuda.synchronize()
if not torch.isfinite(c).all().item():
    raise RuntimeError('CUDA tensor matmul produced non-finite values. The runtime is not healthy enough for model execution.')

print('GPU environment OK')

## 6. Import smoke test

In [ ]:
import subprocess

result = subprocess.run(
    ['bash', '-lc', "PYTHONPATH=src python3 -c \"import visual_memory_benchmark.run; print('import ok')\""],
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError('Import smoke test failed before dataset/model downloads:\n' + result.stderr)

In [ ]:
import json
from pathlib import Path

cfg = json.loads(Path(CONFIG_PATH).read_text())
method_kinds = [item['kind'] for item in cfg['methods']]
if method_kinds != ['text_only_real', 'visual_latent_real']:
    raise RuntimeError(f'Expected real smoke config only, but found methods {method_kinds}')
print('Config methods OK:', method_kinds)

## 7. Prepare small COCO subset

In [ ]:
import subprocess
from pathlib import Path

subset_json = Path(COCO_OUTPUT_DIR) / 'annotations' / 'instances_val2017_subset.json'
if subset_json.exists():
    print('Reusing existing COCO subset:', subset_json)
else:
    prepare = subprocess.run([
        'python3', 'scripts/prepare_coco_subset.py',
        '--output-dir', COCO_OUTPUT_DIR,
        '--num-samples', str(SMOKE_SAMPLE_COUNT),
        '--min-annotations', '3',
        '--min-categories', '2',
    ], text=True)
    if prepare.returncode != 0:
        raise RuntimeError('COCO subset preparation failed.')
print('COCO subset ready')

## 8. Run natural-image smoke config

In [ ]:
import os
import subprocess

env = os.environ.copy()
if HF_TOKEN:
    env['HF_TOKEN'] = HF_TOKEN

run = subprocess.run(['bash', '-lc', f'PYTHONPATH=src python3 -m visual_memory_benchmark.run --config {CONFIG_PATH}'], env=env, text=True)
if run.returncode != 0:
    raise RuntimeError('Natural-image smoke config failed.')
print('Smoke run finished')

## 9. Validate outputs

In [ ]:
from pathlib import Path

run_dir = Path('outputs/natural_coco_smoke')
if not run_dir.exists():
    raise RuntimeError(f'Expected output directory {run_dir} does not exist.')

targets = [
    run_dir / 'aggregate_metrics.csv',
    run_dir / 'per_scene_metrics.csv',
    run_dir / 'comparison.html',
    run_dir / 'pareto_curves.png',
]
for target in targets:
    print(target, target.exists(), target.stat().st_size if target.exists() else 'missing')

for name in ['reconstructions', 'qa', 'artifacts', 'originals']:
    path = run_dir / name
    files = sorted(path.rglob('*')) if path.exists() else []
    real_files = [item for item in files if item.is_file()]
    print(name, 'exists=', path.exists(), 'file_count=', len(real_files))
    for item in real_files[:10]:
        print(' ', item, item.stat().st_size)


## 10. Optional export

In [ ]:
import shutil
from pathlib import Path

run_dir = Path('outputs/natural_coco_smoke')
archive_base = str(EXPORT_ZIP_PATH).removesuffix('.zip')
archive_path = shutil.make_archive(archive_base, 'zip', root_dir=run_dir)
print('Created export zip:', archive_path, Path(archive_path).stat().st_size)

## Optional manual cell: full MVP after smoke passes

Do not run this until the smoke test above has passed in the same session.

```bash
PYTHONPATH=src python3 -m visual_memory_benchmark.run --config configs/natural_coco_mvp.json
```